# Coffee Leaf Disease Classification — Lightweight Baseline (EfficientNet-B0 + ELM) — Source Code

**Author:** Kambale Muhesi Muyisa


# 🌿 Coffee Leaf Disease Classification
## EfficientNet-B0 (Frozen, GPU) + HSV/GLCM/LBP + ELM

| | |
|---|---|
| **Dataset** | `coffee-leaf-disease-dataset/dataset/Train` & `test` |
| **Classes** | Healthy · Miner · Phoma · Rust |
| **Backbone** | EfficientNet-B0 pretrained (frozen, GPU inference) |
| **Features** | 1280-dim deep + 368-dim handcrafted = **1648-dim** |
| **Classifier** | Extreme Learning Machine (ELM) |

## 0. Setup & GPU Check

In [ ]:
# Optional extras
# !pip install scikit-image -q

# ─── Mount Google Drive (Google Colab) ─────────────────────
# The datasets live in your Google Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Not running on Colab (or Drive already mounted):', e)


## 1. Imports & Configuration

In [ ]:
import os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

import cv2
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)

# ─── SEED ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─── DEVICE ──────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🔧 Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ─── PATHS ───────────────────────────────────────────────────
# Datasets are stored in your Google Drive. Edit DRIVE_ROOT to the folder that
# contains the coffee dataset (with its Train/test sub-folders).
DRIVE_ROOT   = Path('/content/drive/MyDrive/coffee_datasets')
DATASET_ROOT = DRIVE_ROOT / 'coffee-leaf-disease-dataset' / 'dataset'
TRAIN_DIR    = DATASET_ROOT / 'Train'
TEST_DIR     = DATASET_ROOT / 'test'

# ─── CONFIG ──────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 64     # increased because a GPU is available

CLASS_FOLDERS = ['Healthy', 'Miner', 'Phoma', 'Rust']
CLASS_VI      = {'Healthy': 'Healthy', 'Miner': 'Leaf Miner',
                 'Phoma':   'Phoma',      'Rust':  'Rust'}
COLORS        = ['#4CAF50', '#FF9800', '#795548', '#F44336']

print('✅ Imports & configuration done!')

## 2. Dataset Inspection & Statistics

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def scan_split(root):
    """Scan one split; return a list of (path, label) and a stats dict."""
    samples, stats = [], {}
    for cls in CLASS_FOLDERS:
        folder = root / cls
        if not folder.exists():
            print(f'  ⚠️  Not found: {folder}')
            stats[cls] = 0; continue
        imgs = sorted([p for p in folder.iterdir()
                       if p.suffix.lower() in IMG_EXTS])
        stats[cls] = len(imgs)
        samples.extend([(p, cls) for p in imgs])
    return samples, stats

train_samples, train_stats = scan_split(TRAIN_DIR)
test_samples,  test_stats  = scan_split(TEST_DIR)

print('\n📊 Dataset statistics:')
print(f'{"Class":<10} {"Disease name":<16} {"Train":>7} {"Test":>7} {"Total":>7}')
print('─'*50)
for cls in CLASS_FOLDERS:
    tr = train_stats.get(cls,0); te = test_stats.get(cls,0)
    print(f'{cls:<10} {CLASS_VI[cls]:<16} {tr:>7} {te:>7} {tr+te:>7}')
print('─'*50)
tr_tot = sum(train_stats.values()); te_tot = sum(test_stats.values())
print(f'{"TOTAL":<10} {"":16} {tr_tot:>7} {te_tot:>7} {tr_tot+te_tot:>7}')

In [ ]:
# ── Distribution charts + real sample images ──────────────────
fig = plt.figure(figsize=(16, 8))
gs  = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.3)

# Bar charts
for col, (title, stats) in enumerate([('Train', train_stats), ('Test', test_stats)]):
    ax = fig.add_subplot(gs[0, col*2 : col*2+2])
    counts = [stats.get(c,0) for c in CLASS_FOLDERS]
    bars   = ax.bar([CLASS_VI[c] for c in CLASS_FOLDERS], counts,
                    color=COLORS, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{title} set distribution', fontweight='bold', fontsize=12)
    ax.set_ylim(0, max(counts)*1.3)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                str(cnt), ha='center', fontweight='bold')
    ax.tick_params(axis='x', rotation=15)

# Sample images
for i, cls in enumerate(CLASS_FOLDERS):
    ax  = fig.add_subplot(gs[1, i])
    img = Image.open(next((TRAIN_DIR/cls).iterdir())).convert('RGB').resize((224,224))
    ax.imshow(img)
    ax.set_title(f'{CLASS_VI[cls]}\n({cls})', fontweight='bold', fontsize=10)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor(COLORS[i]); spine.set_linewidth(3)

plt.suptitle('DEV Coffee Dataset — Statistics & Sample Images', fontsize=14, fontweight='bold')
plt.savefig('dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Deep Feature Extraction — EfficientNet-B0 on GPU

In [ ]:
# ── 3.1 Load EfficientNet-B0 (frozen) ─────────────────────────
weights  = EfficientNet_B0_Weights.IMAGENET1K_V1
backbone = efficientnet_b0(weights=weights)
backbone.classifier = nn.Identity()   # drop FC -> output 1280-dim
backbone = backbone.to(DEVICE).eval()
for p in backbone.parameters():
    p.requires_grad = False

print(f'✅ EfficientNet-B0 loaded on {DEVICE}')
print(f'   Params  : ~5.3M (frozen)')
print(f'   Output  : 1280-dim')

# ── 3.2 Dataset class for the DataLoader ─────────────────────
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class CoffeeDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, str(path)

# ── 3.3 Batch inference function on GPU ──────────────────────
@torch.no_grad()
def extract_deep_features_batch(samples, desc='Extracting'):
    """
    Extract deep features for all samples using a DataLoader.
    Runs on GPU if available, automatically falls back to CPU.
    Returns: np.ndarray (N, 1280), list of labels, list of paths
    """
    ds     = CoffeeDataset(samples, transform=transform)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=(DEVICE.type=='cuda'))
    all_feats, all_labels, all_paths = [], [], []
    for imgs, labels, paths in tqdm(loader, desc=desc):
        imgs  = imgs.to(DEVICE)
        feats = backbone(imgs)          # (B, 1280)
        all_feats.append(feats.cpu().numpy())
        all_labels.extend(labels)
        all_paths.extend(paths)
    return np.concatenate(all_feats, axis=0), all_labels, all_paths

print('\n⏳ Extracting deep features (TRAIN)...')
t0 = time.time()
X_train_deep, y_train, paths_train = extract_deep_features_batch(
    train_samples, desc='Train deep')
print(f'   ✅ Done {time.time()-t0:.1f}s  |  shape: {X_train_deep.shape}')

print('⏳ Extracting deep features (TEST)...')
t0 = time.time()
X_test_deep, y_test, paths_test = extract_deep_features_batch(
    test_samples, desc='Test  deep')
print(f'   ✅ Done {time.time()-t0:.1f}s  |  shape: {X_test_deep.shape}')

## 4. Handcrafted Feature Extraction (368-dim)
HSV Histogram (96) + GLCM Haralick (16) + LBP (256)

In [ ]:
def hsv_histogram(img_rgb, bins=32):
    """HSV histogram — 96-dim. Includes the soil background (cultivation context)."""
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    h   = cv2.calcHist([hsv],[0],None,[bins],[0,180]).flatten()
    s   = cv2.calcHist([hsv],[1],None,[bins],[0,256]).flatten()
    v   = cv2.calcHist([hsv],[2],None,[bins],[0,256]).flatten()
    feat = np.concatenate([h,s,v])
    return feat / (feat.sum() + 1e-8)   # (96,)

def glcm_features(gray):
    """GLCM Haralick — 16-dim. 4 directions x 4 properties."""
    glcm = graycomatrix(gray, distances=[1],
                        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    feats = []
    for prop in ['contrast','correlation','energy','homogeneity']:
        feats.extend(graycoprops(glcm, prop).flatten())
    return np.array(feats)   # (16,)

def lbp_histogram(gray, P=8, R=1, bins=256):
    """LBP histogram — 256-dim. Illumination-invariant."""
    lbp  = local_binary_pattern(gray, P, R, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=bins, range=(0,bins))
    return hist / (hist.sum() + 1e-8)   # (256,)

def extract_handcrafted(img_path):
    """Handcrafted pipeline for one image -> 368-dim."""
    img      = Image.open(img_path).convert('RGB').resize((IMG_SIZE,IMG_SIZE))
    img_rgb  = np.array(img)
    img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    return np.concatenate([
        hsv_histogram(img_rgb),   # 96
        glcm_features(img_gray),  # 16
        lbp_histogram(img_gray),  # 256
    ])   # (368,)

def extract_handcrafted_all(paths, desc='HC features'):
    return np.array([extract_handcrafted(p) for p in tqdm(paths, desc=desc)])

print('⏳ Extracting handcrafted features (TRAIN)...')
t0 = time.time()
X_train_hc = extract_handcrafted_all(paths_train, 'Train HC')
print(f'   ✅ Done {time.time()-t0:.1f}s  |  shape: {X_train_hc.shape}')

print('⏳ Extracting handcrafted features (TEST)...')
t0 = time.time()
X_test_hc = extract_handcrafted_all(paths_test, 'Test  HC')
print(f'   ✅ Done {time.time()-t0:.1f}s  |  shape: {X_test_hc.shape}')

## 5. Feature Fusion

In [ ]:
# L2-normalize each group before concatenating
X_train_deep_n = normalize(X_train_deep, norm='l2')
X_train_hc_n   = normalize(X_train_hc,   norm='l2')
X_test_deep_n  = normalize(X_test_deep,  norm='l2')
X_test_hc_n    = normalize(X_test_hc,    norm='l2')

X_train = np.concatenate([X_train_deep_n, X_train_hc_n], axis=1)  # (N, 1648)
X_test  = np.concatenate([X_test_deep_n,  X_test_hc_n],  axis=1)

# Encode labels
le = LabelEncoder().fit(CLASS_FOLDERS)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

print('✅ Fusion complete!')
print(f'   X_train : {X_train.shape}  (1280 deep + 368 handcrafted)')
print(f'   X_test  : {X_test.shape}')
print(f'   Classes : {le.classes_}')

## 6. Extreme Learning Machine (ELM)

In [ ]:
class ELM:
    """
    Extreme Learning Machine — SLFN with a closed-form solution.

    Formulas:
      H   = activation(X @ W^T + b)         # W, b random, fixed
      β   = (H^T H + I/C)^{-1} H^T T        # Moore-Penrose pseudo-inverse
      ŷ   = softmax(H @ β)

    Advantages:
      • Extremely fast training (ms)
      • No local minima
      • Generalizes well on small datasets
    """
    def __init__(self, n_hidden=2000, activation='relu', C=1.0, random_state=42):
        self.n_hidden     = n_hidden
        self.activation   = activation
        self.C            = C
        self.random_state = random_state
        self.W = self.b = self.beta = self.classes_ = None

    def _act(self, X):
        if self.activation == 'relu':    return np.maximum(0, X)
        if self.activation == 'sigmoid': return 1/(1+np.exp(-np.clip(X,-500,500)))
        if self.activation == 'tanh':    return np.tanh(X)
        return X

    def _H(self, X):
        return self._act(X @ self.W.T + self.b)   # (N, n_hidden)

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        self.classes_ = np.unique(y)
        n_cls = len(self.classes_)

        # Random initialization & fixed
        self.W = rng.randn(self.n_hidden, X.shape[1]) * 0.5
        self.b = rng.randn(1, self.n_hidden) * 0.5

        # One-hot
        T = np.zeros((len(y), n_cls))
        for i, c in enumerate(self.classes_):
            T[y == c, i] = 1

        H = self._H(X)   # (N, n_hidden)
        I = np.eye(self.n_hidden)
        self.beta = np.linalg.solve(
            H.T @ H + I / self.C,
            H.T @ T
        )   # faster than pinv
        return self

    def predict_proba(self, X):
        S = self._H(X) @ self.beta
        e = np.exp(S - S.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

print('✅ ELM class ready!')

## 7. Grid Search — Find Optimal Hyperparameters

In [ ]:
C_grid      = [0.01, 0.1, 1, 10, 100]
hidden_grid = [500, 1000, 2000, 3000]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid_results = []
best_acc, best_C, best_H = 0, 1, 1000

print('⏳ Grid Search (5-fold CV) ...')
total = len(C_grid) * len(hidden_grid)
pbar  = tqdm(total=total, desc='Grid Search')

for n_h in hidden_grid:
    for C in C_grid:
        fold_accs = []
        for tr_idx, vl_idx in kf.split(X_train, y_train_enc):
            elm = ELM(n_hidden=n_h, C=C, random_state=SEED)
            elm.fit(X_train[tr_idx], y_train_enc[tr_idx])
            pred = elm.predict(X_train[vl_idx])
            fold_accs.append(accuracy_score(y_train_enc[vl_idx], pred))
        mean_acc = np.mean(fold_accs)
        grid_results.append({'n_hidden': n_h, 'C': C, 'cv_acc': mean_acc})
        if mean_acc > best_acc:
            best_acc, best_C, best_H = mean_acc, C, n_h
        pbar.set_postfix({'n_h': n_h, 'C': C, 'acc': f'{mean_acc:.4f}'})
        pbar.update(1)
pbar.close()

print(f'\n✅ Grid Search done!')
print(f'   Best n_hidden = {best_H}')
print(f'   Best C        = {best_C}')
print(f'   CV Accuracy   = {best_acc:.4f}')

In [ ]:
# Heatmap Grid Search
df_grid = pd.DataFrame(grid_results)
pivot   = df_grid.pivot(index='n_hidden', columns='C', values='cv_acc')

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGn', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'CV Accuracy'})
ax.set_title('Grid Search — CV Accuracy (n_hidden × C)', fontweight='bold', fontsize=12)
ax.set_xlabel('Regularization C')
ax.set_ylabel('Hidden neurons')

# Frame the best cell
r = list(hidden_grid).index(best_H)
c = list(C_grid).index(best_C)
ax.add_patch(plt.Rectangle((c,r),1,1,fill=False,edgecolor='red',lw=3))
ax.text(c+0.5, r+0.5, '★', ha='center', va='center',
        fontsize=16, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('grid_search.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Train ELM with Optimal Hyperparameters

In [ ]:
print(f'⏳ Training ELM (n_hidden={best_H}, C={best_C}) on the full training set...')
t0  = time.time()
elm = ELM(n_hidden=best_H, C=best_C, random_state=SEED)
elm.fit(X_train, y_train_enc)
t_train = time.time() - t0
print(f'✅ Training done in {t_train*1000:.1f} ms!')

## 9. Evaluate Results

In [ ]:
y_pred_enc = elm.predict(X_test)
y_pred     = le.inverse_transform(y_pred_enc)
y_true     = np.array(y_test)

acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average='macro')
pre = precision_score(y_true, y_pred, average='macro')
rec = recall_score(y_true, y_pred, average='macro')

print('═'*52)
print('       RESULTS — TEST SET (DEV Coffee Dataset)')
print('═'*52)
print(f'  Accuracy       : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision Macro: {pre:.4f}')
print(f'  Recall Macro   : {rec:.4f}')
print(f'  F1 Macro       : {f1:.4f}')
print(f'  Train time     : {t_train*1000:.1f} ms (ELM)')
print('═'*52)
print()

target_names = [CLASS_VI[c] for c in le.classes_]
print(classification_report(y_true, y_pred, target_names=target_names))

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────
cm      = confusion_matrix(y_true, y_pred, labels=le.classes_)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in zip(
    axes,
    [cm, cm_norm],
    ['d', '.2f'],
    ['Confusion Matrix (counts)', 'Confusion Matrix (normalized)']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                xticklabels=target_names, yticklabels=target_names,
                linewidths=0.5)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle(f'EfficientNet-B0 + ELM  |  Acc={acc:.4f}  F1={f1:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Per-class Precision / Recall / F1 ─────────────────────────
prec_cls = precision_score(y_true, y_pred, average=None, labels=le.classes_)
rec_cls  = recall_score   (y_true, y_pred, average=None, labels=le.classes_)
f1_cls   = f1_score       (y_true, y_pred, average=None, labels=le.classes_)

x, w = np.arange(len(le.classes_)), 0.25
fig, ax = plt.subplots(figsize=(11, 5))
for i, (vals, label, color) in enumerate([
    (prec_cls, 'Precision', '#2196F3'),
    (rec_cls,  'Recall',    '#4CAF50'),
    (f1_cls,   'F1-Score',  '#FF9800'),
]):
    bars = ax.bar(x + (i-1)*w, vals, w, label=label, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(target_names, fontsize=11)
ax.set_ylim(0, 1.2); ax.set_ylabel('Score')
ax.set_title('Per-class metrics', fontweight='bold', fontsize=13)
ax.axhline(y=acc, color='red', linestyle='--', alpha=0.6,
           label=f'Overall Acc = {acc:.3f}')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Comparison with Baselines

In [ ]:
compare = {}

# Baseline 1: Deep only + SVM
svm1 = SVC(kernel='rbf', C=10, random_state=SEED)
t0 = time.time(); svm1.fit(X_train_deep_n, y_train_enc); t1=time.time()-t0
p1 = le.inverse_transform(svm1.predict(X_test_deep_n))
compare['EfficientNet + SVM'] = {
    'acc': accuracy_score(y_true,p1),
    'f1':  f1_score(y_true,p1,average='macro'),
    'ms':  t1*1000
}

# Baseline 2: Handcrafted only + SVM
svm2 = SVC(kernel='rbf', C=10, random_state=SEED)
t0 = time.time(); svm2.fit(X_train_hc_n, y_train_enc); t2=time.time()-t0
p2 = le.inverse_transform(svm2.predict(X_test_hc_n))
compare['Handcrafted + SVM'] = {
    'acc': accuracy_score(y_true,p2),
    'f1':  f1_score(y_true,p2,average='macro'),
    'ms':  t2*1000
}

# Baseline 3: Combined + SVM
svm3 = SVC(kernel='rbf', C=10, random_state=SEED)
t0 = time.time(); svm3.fit(X_train, y_train_enc); t3=time.time()-t0
p3 = le.inverse_transform(svm3.predict(X_test))
compare['Combined + SVM'] = {
    'acc': accuracy_score(y_true,p3),
    'f1':  f1_score(y_true,p3,average='macro'),
    'ms':  t3*1000
}

# Proposed
compare['Combined + ELM (proposed)'] = {'acc': acc, 'f1': f1, 'ms': t_train*1000}

# Results table
print(f'{"Method":<25} {"Accuracy":>10} {"F1 Macro":>10} {"Train(ms)":>12}')
print('─'*60)
for m, r in compare.items():
    print(f'{m:<25} {r["acc"]:>10.4f} {r["f1"]:>10.4f} {r["ms"]:>10.1f}ms')

In [ ]:
# Comparison chart
methods = list(compare.keys())
accs_c  = [compare[m]['acc'] for m in methods]
f1s_c   = [compare[m]['f1']  for m in methods]
bar_colors = ['#90CAF9','#A5D6A7','#FFCC80','#EF9A9A']

x, w = np.arange(len(methods)), 0.35
fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x-w/2, accs_c, w, label='Accuracy', color=bar_colors, alpha=0.9)
b2 = ax.bar(x+w/2, f1s_c,  w, label='F1 Macro', color=bar_colors, alpha=0.5,
            edgecolor='grey', linewidth=0.8)
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=10, ha='right', fontsize=10)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Comparison of coffee leaf disease classification methods', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.axvspan(len(methods)-1.5, len(methods)-0.5, alpha=0.07, color='gold')
ax.text(len(methods)-1, 1.11, '⭐ Proposed', ha='center', color='darkorange',
        fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. t-SNE — Feature Space Visualization

In [ ]:
from sklearn.manifold import TSNE

perp = min(30, len(X_test)-1)
color_map = dict(zip(CLASS_FOLDERS, COLORS))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sets = [
    (X_test_deep_n, 'Deep Features (1280-dim)'),
    (X_test_hc_n,   'Handcrafted Features (368-dim)'),
    (X_test,        'Combined Features (1648-dim, proposed)'),
]
for ax, (Xv, title) in zip(axes, sets):
    emb = TSNE(n_components=2, random_state=SEED,
               perplexity=perp, n_iter=1000).fit_transform(Xv)
    for cls in CLASS_FOLDERS:
        mask = np.array(y_test) == cls
        ax.scatter(emb[mask,0], emb[mask,1], c=color_map[cls],
                   label=CLASS_VI[cls], alpha=0.8, s=40,
                   edgecolors='white', linewidths=0.4)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8); ax.axis('off')

plt.suptitle('t-SNE — Cluster separability of the feature groups',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('tsne.png', dpi=120, bbox_inches='tight')
plt.show()

## 12. Inference Demo — Top-3 Predictions

In [ ]:
def predict_one(img_path):
    """Predict one image -> (pred_class, top3 list)."""
    # Deep
    img_t = transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        deep = backbone(img_t).cpu().numpy()
    # HC
    hand = extract_handcrafted(img_path).reshape(1,-1)
    # Concat
    feat = np.concatenate([normalize(deep,'l2'), normalize(hand,'l2')], axis=1)
    # Predict
    proba   = elm.predict_proba(feat)[0]
    top3_i  = np.argsort(proba)[::-1][:3]
    top3    = [(le.classes_[i], proba[i]) for i in top3_i]
    return le.classes_[top3_i[0]], top3

# Take one random image per class from the test set
demo_paths = [random.choice([p for p,l in test_samples if l==cls])
              for cls in CLASS_FOLDERS]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, img_path in zip(axes, demo_paths):
    true_cls  = Path(img_path).parent.name
    pred_cls, top3 = predict_one(img_path)
    ok = pred_cls == true_cls

    ax.imshow(Image.open(img_path).convert('RGB').resize((224,224)))
    for spine in ax.spines.values():
        spine.set_edgecolor('#4CAF50' if ok else '#F44336')
        spine.set_linewidth(4)

    lines  = [f"True: {CLASS_VI[true_cls]}",
               f"Pred: {CLASS_VI[pred_cls]} {'✅' if ok else '❌'}",
               "─────────────"]
    lines += [f"{CLASS_VI[c]}: {p:.1%}" for c,p in top3]
    ax.set_title('\n'.join(lines), fontsize=8.5)
    ax.axis('off')

plt.suptitle('Inference Demo — Top-3 Predictions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('inference_demo.png', dpi=120, bbox_inches='tight')
plt.show()

## 13. Results Summary

In [ ]:
W = 62
def _row(s=''):
    return '\u2551' + s.ljust(W) + '\u2551'
print('\u2554' + '\u2550'*W + '\u2557')
print('\u2551' + 'COFFEE LEAF DISEASE CLASSIFICATION \u2014 RESULTS'.center(W) + '\u2551')
print('\u2560' + '\u2550'*W + '\u2563')
print(_row(f'  Dataset        : DEV-coffee-dataset (4 disease classes)'))
print(_row(f'  Train/Test     : {tr_tot}/{te_tot} images'))
print('\u2560' + '\u2550'*W + '\u2563')
print(_row(f'  Backbone       : EfficientNet-B0 (frozen, GPU)'))
print(_row(f'  Deep features  : 1,280 dims'))
print(_row(f'  HC features    : 368 dims (HSV-96 + GLCM-16 + LBP-256)'))
print(_row(f'  Total vector   : 1,648 dims (L2-normalized)'))
print(_row(f'  Classifier     : ELM (n_hidden={best_H}, C={best_C})'))
print('\u2560' + '\u2550'*W + '\u2563')
print(_row(f'  Accuracy       : {acc:.4f}  ({acc*100:.2f}%)'))
print(_row(f'  Precision Macro: {pre:.4f}'))
print(_row(f'  Recall Macro   : {rec:.4f}'))
print(_row(f'  F1 Macro       : {f1:.4f}'))
print(_row(f'  ELM Train time : {t_train*1000:.1f} ms'))
print('\u255a' + '\u2550'*W + '\u255d')
